# Citation Index API: v0.3.1 executable end-to-end guide

This notebook is both a tutorial and a final smoke test for the Docker deployment. It runs the complete supported workflow:

1. upload a PDF to the external **MinerU** extractor,
2. ask **Qwen3.6** to identify the bibliography entries,
3. ask **Qwen3.6** to convert those entries into structured records, and
4. link a bounded sample of the parsed citations to external scholarly indexes.

Each API operation is asynchronous: submission returns a `job_id`, the client polls `/jobs/{job_id}/status`, and the result is read from `/jobs/{job_id}`. The notebook intentionally fails fast if a service is unavailable or a stage returns an invalid result.

> The combined `/process/references` route is not currently enabled, so this notebook demonstrates the supported four-stage composition explicitly.

## Goal

A successful top-to-bottom run proves all of the following:

- the API, Redis, storage, and RQ workers are available;
- Docker containers can reach the host-side MinerU port-forward;
- prompt files are present in the image;
- Qwen returns answer content and valid structured JSON;
- the citation-linking queue returns the requested index IDs and DOIs when found; and
- job failures and missing job IDs are surfaced as errors rather than empty successes.

## Setup

Start MinerU in one terminal. This command remains attached while the port-forward is active:

```bash
oc port-forward service/mineru-api 8000:8000
```

Start the lightweight Docker stack in a second terminal from the repository root:

```bash
docker compose up -d --build --scale worker-llm=1 \
  redis api worker-default worker-llm worker-linking
docker compose ps
```

The Citation Index API is published on host port **8001** because port **8000** is reserved for MinerU. Docker Compose reads the Qwen endpoint and API key from `.env`; never put credentials in this notebook.

In [4]:
import json
import os
import time
import uuid
from pathlib import Path

import requests

### Configuration

Defaults target the local Docker stack, a small repository fixture, and OpenAlex linking for at most three extracted references. Override them through environment variables when needed. Set `CITATION_INDEX_LINKING_TARGETS` to `all` or a comma-separated list containing `openalex`, `matilda`, and/or `wikidata`. OpenCitations is not supported.

```bash
export CITATION_INDEX_API=http://localhost:8001
export CITATION_INDEX_PDF=/absolute/path/to/paper.pdf
export CITATION_INDEX_LINKING_TARGETS=openalex
export CITATION_INDEX_LINKING_LIMIT=3
```

In [5]:
API_BASE = os.getenv("CITATION_INDEX_API", "https://citation-index-api-graphia-app1-staging.apps.bst2.paas.psnc.pl").rstrip("/")
EXPECTED_VERSION = "0.3.1"
PDF_PATH = Path(
    os.getenv("CITATION_INDEX_PDF", "../benchmarks/excite/all_pdfs/44404.pdf")
).expanduser().resolve()
LINKING_TARGETS = os.getenv("CITATION_INDEX_LINKING_TARGETS", "openalex")
MAX_LINKING_REFERENCES = int(os.getenv("CITATION_INDEX_LINKING_LIMIT", "3"))

POLL_INTERVAL_SECONDS = 2
MAX_JOB_WAIT_SECONDS = 1_800
HTTP_TIMEOUT = (10, 120)  # connect timeout, response timeout

session = requests.Session()
print(
    {
        "api": API_BASE,
        "pdf": str(PDF_PATH),
        "pdf_exists": PDF_PATH.is_file(),
        "linking_targets": LINKING_TARGETS,
        "linking_limit": MAX_LINKING_REFERENCES,
    }
)
assert PDF_PATH.is_file(), f"PDF not found: {PDF_PATH}"
assert MAX_LINKING_REFERENCES > 0, "CITATION_INDEX_LINKING_LIMIT must be positive"

{'api': 'https://citation-index-api-graphia-app1-staging.apps.bst2.paas.psnc.pl', 'pdf': '/Users/alex/docs/code/Odoma/citation_index/benchmarks/excite/all_pdfs/44404.pdf', 'pdf_exists': True, 'linking_targets': 'openalex', 'linking_limit': 3}


## Step 1 — Verify service health

`GET /health` checks the API's Redis and storage dependencies. It does not call MinerU or Qwen, so those integrations are tested by the later stages.

In [6]:
health_response = session.get(f"{API_BASE}/health", timeout=HTTP_TIMEOUT)
health_response.raise_for_status()
health = health_response.json()
print(health)
assert health["status"] == "healthy", health
assert health["redis"] == "ok" and health["storage"] == "ok", health
assert health["version"] == EXPECTED_VERSION, health

{'status': 'healthy', 'redis': 'ok', 'storage': 'ok', 'version': '0.3.1'}


## Step 2 — Understand the asynchronous job contract

Submission endpoints return HTTP 200 with a job object such as:

```json
{"job_id": "...", "status": "queued", "created_at": "..."}
```

A job then moves through `queued` and `processing` to either `completed` or `failed`. The helper below prints only state changes, raises with the server error on failure, and enforces a maximum wait.

In [7]:
def response_json(response: requests.Response) -> dict:
    """Raise an informative HTTP error, then return a JSON object."""
    try:
        payload = response.json()
    except requests.JSONDecodeError as exc:
        preview = response.text[:300].replace("\n", " ")
        raise RuntimeError(
            f"Expected JSON from {response.request.method} {response.url}; "
            f"HTTP {response.status_code}, body={preview!r}"
        ) from exc
    if not response.ok:
        raise RuntimeError(
            f"{response.request.method} {response.url} failed with "
            f"HTTP {response.status_code}: {payload}"
        )
    if not isinstance(payload, dict):
        raise TypeError(f"Expected a JSON object, got {type(payload).__name__}")
    return payload


def wait_for_job(job_id: str, max_wait: int = MAX_JOB_WAIT_SECONDS) -> dict:
    """Poll a job to a terminal state and return its persisted result."""
    started = time.monotonic()
    last_state = None

    while time.monotonic() - started < max_wait:
        status_response = session.get(
            f"{API_BASE}/jobs/{job_id}/status", timeout=HTTP_TIMEOUT
        )
        status_payload = response_json(status_response)
        state = status_payload["status"]

        if state != last_state:
            elapsed = time.monotonic() - started
            stage = status_payload.get("current_stage")
            print(f"[{elapsed:6.1f}s] {job_id}: {state} (stage={stage})")
            last_state = state

        if state == "completed":
            return response_json(
                session.get(f"{API_BASE}/jobs/{job_id}", timeout=HTTP_TIMEOUT)
            )
        if state == "failed":
            raise RuntimeError(
                f"Job {job_id} failed: {status_payload.get('error', 'unknown error')}"
            )

        time.sleep(POLL_INTERVAL_SECONDS)

    raise TimeoutError(f"Job {job_id} did not finish within {max_wait}s")

## Step 3 — Extract Markdown with MinerU

`POST /extract/text` accepts a PDF as multipart form data. Supported extractor names are `pymupdf`, `mineru`, and `grobid`; Marker has been removed. This test selects `mineru`, which makes a real call through `host.docker.internal:8000` to the OpenShift port-forward.

In [8]:
with PDF_PATH.open("rb") as pdf_file:
    submit_text_response = session.post(
        f"{API_BASE}/extract/text",
        params={"extractor": "mineru", "markdown": "true"},
        files={"file": (PDF_PATH.name, pdf_file, "application/pdf")},
        timeout=HTTP_TIMEOUT,
    )

text_job = response_json(submit_text_response)
print(text_job)
assert text_job["status"] in {"queued", "processing"}

{'job_id': 'b7d58d8b-e34e-45cc-b331-49f0bf415e42', 'status': 'queued', 'created_at': '2026-09-11T13:57:54.822682', 'message': 'Text extraction job enqueued'}


In [9]:
text_result = wait_for_job(text_job["job_id"])
extracted_text = text_result.get("text", "")

assert text_result.get("extractor") == "mineru", text_result
assert len(extracted_text) > 1_000, "MinerU returned unexpectedly little text"

print(
    {
        "extractor": text_result.get("extractor"),
        "backend": text_result.get("backend"),
        "mineru_version": text_result.get("mineru_version"),
        "text_characters": len(extracted_text),
    }
)
print("\nPreview:\n", extracted_text[:700].strip())

[   0.0s] b7d58d8b-e34e-45cc-b331-49f0bf415e42: queued (stage=None)
[   2.1s] b7d58d8b-e34e-45cc-b331-49f0bf415e42: processing (stage=text_extraction)
[  12.6s] b7d58d8b-e34e-45cc-b331-49f0bf415e42: completed (stage=text_extraction)
{'extractor': 'mineru', 'backend': None, 'mineru_version': None, 'text_characters': 32675}

Preview:
 # BRENNPUNKT LATEINAMERIKA

POLITIK · WIRTSCHAFT · GESELLSCHAFT

INSTITUT FÜR IBEROAMERIKA-KUNDE HAMBURG

Nummer 12

30. Juni 2000

ISSN 1437-6148

# Die politische Krise in Peru: Festsetzung des Fujimorismo und Polarisierung des Landes

Andreas Steinhauf

Von den umstrittensten und schmutzigsten Wahlen in der Geschichte Perus ist die Rede. Am 28. Mai 2000 wurde der amtierende Präsident Alberto Fujimori von der Obersten Wahlbehörde zum Sieger der Stichwahl erklärt, zu der sein Kontrahent Alejandro Toledo bereits nicht mehr angetreten war und stattdessen die Bevölkerung zum Boykott aufgerufen hatte. Damit wird er am 28. Juli, wenn verfassungsgemäß der neue P

## Step 4 — Extract raw bibliography entries with Qwen

`POST /extract/references` accepts Markdown as JSON. The `full_text` method sends the document to the configured medium-intelligence model (`Qwen3.6-27B` by default). Temperature `0.0` makes this validation run as deterministic as the serving stack permits.

In [10]:
submit_extraction_response = session.post(
    f"{API_BASE}/extract/references",
    params={"method": "full_text", "temperature": 0.0},
    json={"text": extracted_text},
    timeout=HTTP_TIMEOUT,
)

extraction_job = response_json(submit_extraction_response)
print(extraction_job)

{'job_id': '2f071f50-f9d7-4511-9cf7-5468164a4c75', 'status': 'queued', 'created_at': '2026-09-11T13:58:07.710313', 'message': 'Reference extraction job enqueued'}


In [11]:
extraction_result = wait_for_job(extraction_job["job_id"])
raw_references = extraction_result.get("references", [])

assert raw_references, "Qwen returned no references for a document with a bibliography"
assert extraction_result.get("count") == len(raw_references)
assert all(isinstance(item, str) and item.strip() for item in raw_references)

print(f"Extracted {len(raw_references)} bibliography entries:")
for index, reference in enumerate(raw_references[:10], start=1):
    print(f"{index:>2}. {reference[:300]}")

[   0.0s] 2f071f50-f9d7-4511-9cf7-5468164a4c75: queued (stage=None)
[   2.1s] 2f071f50-f9d7-4511-9cf7-5468164a4c75: processing (stage=reference_extraction)
[  10.5s] 2f071f50-f9d7-4511-9cf7-5468164a4c75: completed (stage=reference_extraction)
Extracted 7 bibliography entries:
 1. Resumen Semanal, DESCO, Lima: http://www.desco.org.pe/rs-in.HTM
 2. Que Hacer, DESCO, Lima: http://www.desco.org.pe/qh/qh-in.htm#OH
 3. Caretas: http://www.caretas.com.pe/
 4. El Comercio: http://www.elcomercioperu.com
 5. Ingolf Dietrich. Die Koka- und Kokainwirtschaft Perus. Frankfurt/M.: Vervuert 1998, 314 S., ISBN 3-89354-247-7, Band 48
 6. Peter Thiery. Transformation in Chile. Institutioneller Wandel, Entwicklung und Demokratie 1973-1996. Frankfurt/M.: Vervuert 2000, ca. 354 S., Band 51
 7. Judith Schultz. Präsidentielle Demokratien in Lateinamerika. Eine Untersuchung der präsidentiellen Regierungssysteme von Costa Rica und Venezuela. Frankfurt/M.: Vervuert 2000, ca. 490 S., Band 52


## Step 5 — Parse the entries into structured records

`POST /parse/references` accepts a JSON list of strings. With `parser=llm`, Qwen returns fields such as authors, title, year, publication venue, volume, issue, and identifiers when they are present in the source citation.

In [12]:
submit_parse_response = session.post(
    f"{API_BASE}/parse/references",
    params={"parser": "llm", "temperature": 0.0},
    json={"references": raw_references},
    timeout=HTTP_TIMEOUT,
)

parse_job = response_json(submit_parse_response)
print(parse_job)

{'job_id': '4f486b81-727a-4cd7-8f96-823af3ce3bec', 'status': 'queued', 'created_at': '2026-09-11T13:58:18.326181', 'message': 'Reference parsing job enqueued'}


In [13]:
parse_result = wait_for_job(parse_job["job_id"])
structured_references = parse_result.get("references", [])

assert parse_result.get("parser") == "llm", parse_result
assert parse_result.get("count") == len(structured_references)
assert len(structured_references) == len(raw_references)
assert all(isinstance(item, dict) for item in structured_references)

# Counts alone cannot detect a field-level regression: one record per input string is
# returned even when every title is null. Require most references to carry a title.
MIN_TITLED_SHARE = 0.8

untitled = [
    reference
    for reference in structured_references
    if not (reference.get("full_title") or reference.get("journal_title"))
]
titled_share = 1 - len(untitled) / len(structured_references)
assert titled_share >= MIN_TITLED_SHARE, (
    f"only {titled_share:.0%} of references carry a title, expected at least "
    f"{MIN_TITLED_SHARE:.0%}; untitled entries: {untitled}"
)

print(f"Parsed {len(structured_references)} structured references, {titled_share:.0%} titled.")
print(json.dumps(structured_references[:2], indent=2, ensure_ascii=False)[:4_000])

[   0.0s] 4f486b81-727a-4cd7-8f96-823af3ce3bec: queued (stage=None)
[   2.1s] 4f486b81-727a-4cd7-8f96-823af3ce3bec: processing (stage=reference_parsing)
[  39.6s] 4f486b81-727a-4cd7-8f96-823af3ce3bec: completed (stage=reference_parsing)
Parsed 7 structured references, 100% titled.
[
  {
    "full_title": "Resumen Semanal",
    "journal_title": null,
    "authors": null,
    "editors": null,
    "publisher": "DESCO",
    "translator": null,
    "publication_place": "Lima",
    "publication_year": null,
    "publication_date_raw": null,
    "identifiers": [
      {
        "scheme": "URL",
        "value": "http://www.desco.org.pe/rs-in.HTM",
        "normalized": null
      }
    ],
    "ref_type": "webpage",
    "raw": {},
    "volume": null,
    "issue": null,
    "pages": null,
    "cited_range": null,
    "footnote_number": null
  },
  {
    "full_title": "Que Hacer",
    "journal_title": null,
    "authors": null,
    "editors": null,
    "publisher": "DESCO",
    "translator": nul

## Step 6 — Link citations to scholarly indexes

`POST /link/references` accepts a parsed reference object with required `full_title`, non-empty `authors`, and integer `publication_year` fields. With `batched=true`, `reference` must be a non-empty JSON array of these objects. Raw strings are rejected. Candidates are searched using parsed fields and filtered with `custom_match()`: title similarity ≥90 and either first-author similarity ≥70 or year within ±1. The `target` query parameter accepts `openalex`, `matilda`, `wikidata`, `all`, or a comma-separated list; OpenCitations is intentionally excluded.

This executable check takes the structured records from Step 5, selects records with the required title, first-author name, and year, and submits only those three fields. It reports how many records lack required fields and links at most `MAX_LINKING_REFERENCES` eligible citations to keep live external requests bounded. A target can legitimately return `null` for both fields when it cannot make a confident match. The default target is OpenAlex because Matilda may require deployment-specific credentials.

In [14]:
eligible_references = []
for reference in structured_references:
    title = reference.get("full_title")
    authors = reference.get("authors") or []
    year = reference.get("publication_year")
    first_author = authors[0] if authors else None
    author_name = (
        first_author
        if isinstance(first_author, str)
        else (first_author.get("surname") or first_author.get("name"))
        if isinstance(first_author, dict)
        else None
    )
    if (
        isinstance(title, str) and title.strip()
        and isinstance(author_name, str) and author_name.strip()
        and type(year) is int and 1 <= year <= 9999
    ):
        eligible_references.append({
            "full_title": title,
            "authors": [
                {key: value for key, value in author.items() if value is not None}
                if isinstance(author, dict) else author
                for author in authors
            ],
            "publication_year": year,
        })

print(f"Skipped {len(structured_references) - len(eligible_references)} records missing linking fields")
references_to_link = eligible_references[:MAX_LINKING_REFERENCES]
assert references_to_link, "No parsed references have the required title, author, and year"

submit_link_response = session.post(
    f"{API_BASE}/link/references",
    params={"batched": "true", "target": LINKING_TARGETS},
    json={"reference": references_to_link},
    timeout=HTTP_TIMEOUT,
)

link_job = response_json(submit_link_response)
print(link_job)

Skipped 4 records missing linking fields
{'job_id': '90d0ff2c-0e99-4285-8170-2daaf216f5f2', 'status': 'queued', 'created_at': '2026-09-11T13:58:58.043075', 'message': 'Citation linking job enqueued'}


In [15]:
link_result = wait_for_job(link_job["job_id"])
linked_references = link_result.get("results", [])
requested_targets = [
    target.strip().lower()
    for target in LINKING_TARGETS.split(",")
    if target.strip()
]
expected_linking_targets = (
    ["openalex", "matilda", "wikidata"]
    if "all" in requested_targets
    else list(dict.fromkeys(requested_targets))
)

assert link_result.get("count") == len(references_to_link), link_result
assert link_result.get("targets") == expected_linking_targets, link_result
assert len(linked_references) == len(references_to_link), link_result
assert [item.get("reference") for item in linked_references] == references_to_link
for item in linked_references:
    assert list(item.get("links", {})) == expected_linking_targets, item
    for link in item["links"].values():
        assert set(link) == {"id", "doi"}, link
        assert link["id"] is None or isinstance(link["id"], str), link
        assert link["doi"] is None or isinstance(link["doi"], str), link

linked_id_count = sum(
    link["id"] is not None
    for item in linked_references
    for link in item["links"].values()
)
print(
    json.dumps(
        {"linked_id_count": linked_id_count, "results": linked_references},
        indent=2,
        ensure_ascii=False,
    )
)

[   0.0s] 90d0ff2c-0e99-4285-8170-2daaf216f5f2: queued (stage=None)
[   2.1s] 90d0ff2c-0e99-4285-8170-2daaf216f5f2: processing (stage=citation_linking)
[   4.1s] 90d0ff2c-0e99-4285-8170-2daaf216f5f2: completed (stage=citation_linking)
{
  "linked_id_count": 2,
  "results": [
    {
      "reference": {
        "full_title": "Die Koka- und Kokainwirtschaft Perus",
        "authors": [
          {
            "first_name": "Ingolf",
            "surname": "Dietrich"
          }
        ],
        "publication_year": 1998
      },
      "links": {
        "openalex": {
          "id": "https://openalex.org/W2924573179",
          "doi": "10.31819/9783964566904"
        }
      }
    },
    {
      "reference": {
        "full_title": "Transformation in Chile. Institutioneller Wandel, Entwicklung und Demokratie 1973-1996",
        "authors": [
          {
            "first_name": "Peter",
            "surname": "Thiery"
          }
        ],
        "publication_year": 2000
      },
     

## Checks — verify the error contract

A missing job must return HTTP 404. In production code, use `response_json` or equivalent handling so failed jobs and invalid IDs cannot be mistaken for empty results.

The linking endpoint must reject raw strings and parsed objects missing a required field with HTTP 422 before creating a job.

In [16]:
missing_job_id = str(uuid.uuid4())
missing_response = session.get(
    f"{API_BASE}/jobs/{missing_job_id}/status", timeout=HTTP_TIMEOUT
)
print({"status_code": missing_response.status_code, "body": missing_response.json()})
assert missing_response.status_code == 404

invalid_linking_inputs = ["Smith. A paper. 2024."]
for field in ("full_title", "authors", "publication_year"):
    incomplete_reference = dict(references_to_link[0])
    del incomplete_reference[field]
    invalid_linking_inputs.append(incomplete_reference)

for invalid_reference in invalid_linking_inputs:
    invalid_response = session.post(
        f"{API_BASE}/link/references",
        json={"reference": invalid_reference},
        timeout=HTTP_TIMEOUT,
    )
    assert invalid_response.status_code == 422, invalid_response.text
print("Invalid linking inputs rejected with HTTP 422")

{'status_code': 404, 'body': {'detail': 'Job b831dcc5-20dd-47fe-84e3-cf92a3c7363b not found'}}
Invalid linking inputs rejected with HTTP 422


## Checks — final end-to-end verdict

This cell deliberately contains assertions, not just display logic. If it prints `PASS`, the tested PDF completed every supported stage and all required outputs were persisted.

In [17]:
end_to_end_checks = {
    "api_version": health["version"] == EXPECTED_VERSION,
    "api_healthy": health["status"] == "healthy",
    "mineru_text_returned": len(extracted_text) > 1_000,
    "raw_references_returned": len(raw_references) > 0,
    "all_references_parsed": len(structured_references) == len(raw_references),
    "references_carry_titles": titled_share >= MIN_TITLED_SHARE,
    "all_linking_results_returned": len(linked_references) == len(references_to_link),
    "linking_shape_is_valid": all(
        list(item.get("links", {})) == expected_linking_targets
        for item in linked_references
    ),
    "missing_job_returns_404": missing_response.status_code == 404,
}
assert all(end_to_end_checks.values()), end_to_end_checks

verdict = {
    "verdict": "PASS",
    "version": health["version"],
    "pdf": PDF_PATH.name,
    "text_characters": len(extracted_text),
    "raw_reference_count": len(raw_references),
    "structured_reference_count": len(structured_references),
    "titled_share": round(titled_share, 3),
    "linking_reference_count": len(linked_references),
    "linking_targets": expected_linking_targets,
    "linked_id_count": linked_id_count,
    "jobs": {
        "text_extraction": text_job["job_id"],
        "reference_extraction": extraction_job["job_id"],
        "reference_parsing": parse_job["job_id"],
        "citation_linking": link_job["job_id"],
    },
}
print(json.dumps(verdict, indent=2))

{
  "verdict": "PASS",
  "version": "0.3.1",
  "pdf": "44404.pdf",
  "text_characters": 32675,
  "raw_reference_count": 7,
  "structured_reference_count": 7,
  "titled_share": 1.0,
  "linking_reference_count": 3,
  "linking_targets": [
    "openalex"
  ],
  "linked_id_count": 2,
  "jobs": {
    "text_extraction": "b7d58d8b-e34e-45cc-b331-49f0bf415e42",
    "reference_extraction": "2f071f50-f9d7-4511-9cf7-5468164a4c75",
    "reference_parsing": "4f486b81-727a-4cd7-8f96-823af3ce3bec",
    "citation_linking": "90d0ff2c-0e99-4285-8170-2daaf216f5f2"
  }
}


## Equivalent command-line workflow

The first stage can be submitted with curl:

```bash
curl -F "file=@/absolute/path/paper.pdf;type=application/pdf" \
  "http://localhost:8001/extract/text?extractor=mineru&markdown=true"
```

Poll and retrieve any returned job ID:

```bash
curl http://localhost:8001/jobs/JOB_ID/status
curl http://localhost:8001/jobs/JOB_ID
```

Submit a parsed citation batch to more than one linking target:

```bash
curl -X POST \
  "http://localhost:8001/link/references?batched=true&target=openalex,wikidata" \
  -H "Content-Type: application/json" \
  -d '{"reference":[{"full_title":"Deep learning","authors":["LeCun, Y."],"publication_year":2015}]}'
```

For the other JSON stages, write the previous response to a file or use `jq` to build the next request. Python is usually clearer for the complete chain because extracted document text can be large.

## API quick reference

| Method | Path | Queue | Input | Result |
|---|---|---|---|---|
| `GET` | `/health` | — | — | service health |
| `POST` | `/extract/text` | `default` | multipart PDF | Markdown text |
| `POST` | `/extract/references` | `llm-tasks` | `{"text": "..."}` | raw citation strings |
| `POST` | `/parse/references` | `llm-tasks` or `default` | `{"references": [...]}` | structured records |
| `POST` | `/link/references` | `linking` | `{"reference": {"full_title": "...", "authors": ["Smith"], "publication_year": 2024}}` or object array | target IDs and DOIs |
| `GET` | `/jobs/{job_id}/status` | — | job ID | status and error metadata |
| `GET` | `/jobs/{job_id}` | — | completed job ID | persisted result |

Interactive OpenAPI documentation is available at [`http://localhost:8001/docs`](http://localhost:8001/docs).